# Assignment 1: Weather Agent
    
    Build an AI agent using the **Gemini SDK** that retrieves the current weather for **three specified locations** using OpenWeather. The agent must make the weather API calls **sequentially**, collect the temperature from each location, and calculate and display the **average temperature** across all three locations.
    

```text
                 USER
                  │
                  ▼
          Enter 3 locations
                  │
                  ▼
            GEMINI SDK
             AI Agent
                  │
        ┌─────────┼─────────┐
        │         │         │
        ▼         ▼         ▼
    Location 1 Location 2 Location 3
        │         │         │
        ▼         ▼         ▼
   OpenWeather OpenWeather OpenWeather
        │         │         │
        ▼         ▼         ▼
     Temp 1     Temp 2     Temp 3
        │         │         │
        └─────────┼─────────┘
                  ▼
       Calculate Average
                  │
                  ▼
          Display Result
```


In [1]:
%pip install -q google-genai pyowm python-dotenv

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os

from dotenv import load_dotenv
from google import genai
from google.genai import types
from pyowm import OWM

In [3]:
# Load API keys from the local .env file
load_dotenv()

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
OPENWEATHER_API_KEY = os.getenv("OPENWEATHER_API_KEY")

missing_keys = [
    name for name, value in {
        "GEMINI_API_KEY": GEMINI_API_KEY,
        "OPENWEATHER_API_KEY": OPENWEATHER_API_KEY,
    }.items()
    if not value or value.startswith("your_")
]

if missing_keys:
    raise ValueError(
        "Missing API key(s): " + ", ".join(missing_keys) +
        ". Add real values to a .env file in the workspace folder."
    )

print("Gemini and OpenWeather API keys loaded from .env.")

Gemini and OpenWeather API keys loaded from .env.


In [4]:
# Initialize Gemini
client = genai.Client(api_key=GEMINI_API_KEY)
# Initialize OpenWeather
owm = OWM(OPENWEATHER_API_KEY)
weather_manager = owm.weather_manager()

print("Gemini and OpenWeather initialized successfully.")

Gemini and OpenWeather initialized successfully.


In [5]:
# # Test Gemini SDK
# response = client.models.generate_content(
#     model="gemini-3.6-flash",
#     contents="Explain what an ASML is in exactly two concise sentences."
# )

# print("--- Gemini Test Response ---")
# print(response.text)

In [6]:
# OpenWeather Tool
def get_current_weather(city: str) -> dict:
    """
    Retrieve current weather information for a city
    using OpenWeather.
    """

    try:
        observation = weather_manager.weather_at_place(city)
        weather = observation.weather

        temperature = weather.temperature("celsius")["temp"]

        return {
            "city": city,
            "temperature_celsius": temperature,
            "status": weather.status,
            "humidity": weather.humidity,
            "wind_speed_mps": weather.wind()["speed"]
        }

    except Exception as e:
        return {
            "city": city,
            "error": str(e)
        }

In [7]:
# # Test OpenWeather Tool

# test_city = "Kathmandu"

# result = get_current_weather(test_city)

# if "error" in result:
#     print(f"Error: {result['error']}")
# else:
#     print(f"Weather Data for {result['city']}")
#     print("-" * 40)
#     print(f"Status:        {result['status']}")
#     print(f"Temperature:   {result['temperature_celsius']} °C")
#     print(f"Humidity:      {result['humidity']}%")
#     print(f"Wind Speed:    {result['wind_speed_mps']} m/s")

In [8]:
# Define the Weather Function for Gemini

weather_function = types.FunctionDeclaration(
    name="get_current_weather",
    description="Get the current weather and temperature for a specified city.",
    parameters={
        "type": "OBJECT",
        "properties": {
            "city": {
                "type": "STRING",
                "description": "The name of the city."
            }
        },
        "required": ["city"]
    }
)

weather_tool = types.Tool(
    function_declarations=[weather_function]
)

In [9]:
# Weather Agent Configuration

agent_config = types.GenerateContentConfig(
    system_instruction="""
You are a weather assistant.

The user provides three locations.

For each location:
1. Use the get_current_weather tool.
2. Obtain the current temperature.
3. Do not invent or estimate weather information.

The three locations must be processed sequentially.

After all three temperatures have been obtained:
- Report each location and its temperature.
- Calculate the average temperature.
- Display the average temperature in Celsius.
""",
    tools=[weather_tool]
)

In [10]:
# Get Three Locations from the User
print("Enter three locations for weather data")
print()

locations = []

for i in range(1, 4):
    city = input(f"Enter location {i}: ").strip()
    locations.append(city)

print("\n Selected Locations:")
for i, city in enumerate(locations, start=1):
    print(f"{i}. {city}")

Enter three locations for weather data


 Selected Locations:
1. cambridge
2. boston
3. kathmandu


In [11]:
# Sequential Weather Agent
weather_results = []

for i, city in enumerate(locations, start=1):

    print(f"\n Processing Location {i}: {city}")

    prompt = f"""
Get the current weather for the following location:

{city}

Use the get_current_weather tool.
Return the current temperature.
"""

    response = client.models.generate_content(
        model="gemini-3.6-flash",
        contents=prompt,
        config=agent_config
    )

    # Check whether Gemini requested the weather function
    if response.function_calls:

        function_call = response.function_calls[0]

        if function_call.name == "get_current_weather":

            tool_city = function_call.args["city"]

            # Sequential OpenWeather API Call

            weather_data = get_current_weather(tool_city)

            if "error" in weather_data:
                print(f"Error: {weather_data['error']}")
                continue

            weather_results.append(weather_data)

            print(
                f"{weather_data['city']}: "
                f"{weather_data['temperature_celsius']:.2f} °C"
            )

    else:
        print("Gemini did not request the weather tool.")


 Processing Location 1: cambridge
cambridge: 13.46 °C

 Processing Location 2: boston
boston: 15.03 °C

 Processing Location 3: kathmandu
kathmandu: 16.72 °C


In [12]:
# Calculate Average Temperature

if len(weather_results) == 3:

    temperatures = [
        result["temperature_celsius"]
        for result in weather_results
    ]

    average_temperature = sum(temperatures) / len(temperatures)

    print("\n" + "=" * 50)
    print("FINAL WEATHER SUMMARY")
    print("=" * 50)

    for result in weather_results:
        print(
            f"{result['city']}: "
            f"{result['temperature_celsius']:.2f} °C"
        )

    print("-" * 50)

    print(
        f"Average Temperature: "
        f"{average_temperature:.2f} °C"
    )

else:

    print("\nUnable to retrieve weather for all three locations.")

    print(
        f"Successfully retrieved: "
        f"{len(weather_results)}/3 locations"
    )


FINAL WEATHER SUMMARY
cambridge: 13.46 °C
boston: 15.03 °C
kathmandu: 16.72 °C
--------------------------------------------------
Average Temperature: 15.07 °C
